In [1]:
import numpy as np
import pandas as pd

In [9]:
df = pd.read_csv("IMDB Dataset.csv", engine='python', on_bad_lines='skip')
df = df.iloc[:10000]
df.sample(5)["review"]

,review
2737,"This was very good, except for two things whic..."
8131,"Having already seen the original ""Jack Frost"",..."
2590,I must confess that I was completely shocked b...
5664,"I took a chance on ""Hardcastle and McCormick"" ..."
7641,"This film proves that the ""commercial"" cinema ..."


In [10]:
df["sentiment"].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [11]:
df.isnull().sum()

,0
review,0
sentiment,0


In [12]:
df.duplicated().sum()

np.int64(17)

In [13]:
df.drop_duplicates(inplace=True)

In [14]:
df.duplicated().sum()

np.int64(0)

In [15]:
# Basic Preprocessing
# Remove tags
# lowercase
# remove stopwords

In [16]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [17]:
df['review'] = df['review'].apply(remove_tags)

In [18]:

df['review'] = df['review'].apply(lambda x:x.lower())

In [22]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [23]:
X = df.iloc[:,0:1]
y = df['sentiment']

In [24]:

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [25]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [26]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [29]:

# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [30]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [31]:
X_train_bow.shape

(7986, 48282)

In [32]:

from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(X_train_bow,y_train)

GaussianNB()

In [33]:

y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import accuracy_score,confusion_matrix
accuracy_score(y_test,y_pred)

0.6324486730095142

In [34]:
confusion_matrix(y_test,y_pred)

array([[717, 235],
       [499, 546]])

In [35]:
# Applying RF
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8487731597396094

In [36]:
cv = CountVectorizer(max_features=3000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8422633950926389

In [37]:
# Using Both 1 and 2 Uni + Bi-Gram
cv = CountVectorizer(ngram_range=(1,2),max_features=5000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8342513770655984

# Using TF-IDF


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [39]:

tfidf = TfidfVectorizer()

In [40]:
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review'])

In [41]:
rf = RandomForestClassifier()

rf.fit(X_train_tfidf,y_train)
y_pred = rf.predict(X_test_tfidf)

accuracy_score(y_test,y_pred)

0.8452679018527791

# Applying Word-2-Vec

In [44]:
import gensim.downloader as api

# This downloads the ~1.6GB file and loads the model
# Note: This might take 3-5 minutes depending on Colab's network speed
wv = api.load('word2vec-google-news-300')

# Test it out
print(wv.most_similar('politic'))

[==================================================] 100.0% 1662.8/1662.8MB downloaded
[('Expediency_asks', 0.6325168013572693), ('politik', 0.5434165596961975), ('politcs', 0.5120305418968201), ('politics', 0.5113316178321838), ('politcal', 0.5102152228355408), ('poltical', 0.5066124200820923), ('political', 0.49901166558265686), ('neo_Ottomanism', 0.47701916098594666), ('ocracy', 0.47587379813194275), ('polticial', 0.47480452060699463)]


In [45]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

In [46]:

# Remove stopwords

X_train = X_train['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))
# Remove stopwords

X_test = X_test['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

In [48]:
nltk.download('punkt_tab')
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [49]:

model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [50]:

model.build_vocab(story)

In [51]:

model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5849814, 6186875)

In [52]:
len(model.wv.index_to_key)

31845

In [53]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [54]:
document_vector(df['review'].values[0])

array([-0.05587964,  0.34697703,  0.05608236, -0.10863955,  0.07176287,
       -0.5318095 ,  0.19102298,  0.83684623, -0.22365476, -0.06550234,
       -0.11449907, -0.4973821 ,  0.06211264,  0.22302821,  0.03412394,
       -0.29153278,  0.1407601 , -0.46178564,  0.09174993, -0.7075972 ,
        0.10198315,  0.07974541,  0.22481973, -0.11623806, -0.18793951,
       -0.18720238, -0.13566165, -0.1994122 , -0.3016148 , -0.01909539,
        0.47105357, -0.04700963, -0.01392678, -0.3329528 , -0.3012571 ,
        0.24661113,  0.10423026, -0.38161886, -0.27955577, -0.7074497 ,
        0.04746695, -0.24761301, -0.2698284 , -0.01141151,  0.4628468 ,
       -0.20045619, -0.49048895, -0.05200685,  0.18121506,  0.27681893,
        0.12906781, -0.30946583, -0.3068841 , -0.10147916, -0.32874614,
        0.10782082,  0.33433092,  0.05863902, -0.31158915,  0.0733523 ,
       -0.04897172,  0.08982359, -0.14643303,  0.05745431, -0.34910026,
        0.3678122 , -0.03525482,  0.24122009, -0.63558584,  0.23

In [55]:

from tqdm import tqdm

In [56]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

100%|██████████| 9983/9983 [07:09<00:00, 23.24it/s]


In [57]:

X = np.array(X)
X[0]

array([-0.05587964,  0.34697703,  0.05608236, -0.10863955,  0.07176287,
       -0.5318095 ,  0.19102298,  0.83684623, -0.22365476, -0.06550234,
       -0.11449907, -0.4973821 ,  0.06211264,  0.22302821,  0.03412394,
       -0.29153278,  0.1407601 , -0.46178564,  0.09174993, -0.7075972 ,
        0.10198315,  0.07974541,  0.22481973, -0.11623806, -0.18793951,
       -0.18720238, -0.13566165, -0.1994122 , -0.3016148 , -0.01909539,
        0.47105357, -0.04700963, -0.01392678, -0.3329528 , -0.3012571 ,
        0.24661113,  0.10423026, -0.38161886, -0.27955577, -0.7074497 ,
        0.04746695, -0.24761301, -0.2698284 , -0.01141151,  0.4628468 ,
       -0.20045619, -0.49048895, -0.05200685,  0.18121506,  0.27681893,
        0.12906781, -0.30946583, -0.3068841 , -0.10147916, -0.32874614,
        0.10782082,  0.33433092,  0.05863902, -0.31158915,  0.0733523 ,
       -0.04897172,  0.08982359, -0.14643303,  0.05745431, -0.34910026,
        0.3678122 , -0.03525482,  0.24122009, -0.63558584,  0.23

In [58]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [59]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.771156735102654